# EvolveAI — T4 Multi-Source Streaming Learning\n\nThis notebook runs one bounded SNN training job. A Gatekeeper reads multiple approved Hugging Face datasets in streaming mode, filters and labels records, and feeds a bounded in-memory replay buffer to the SNN. It does not download or merge complete corpora. Training progress prints the source, domain, sample preview, losses, buffer counts and source statistics.\n\n**Important:** this is a controlled research prototype, not a conscious or superintelligent system. Review upstream dataset cards, licenses and content rights before enabling sources. Colab GPU quotas still apply; one invocation does not bypass them.

In [ ]:
from pathlib import Path
import subprocess, os, sys

REPO_DIR = Path('/content/EvolveAI')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/swasthik-s/EvolveAI.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

%cd /content/EvolveAI/snn
%pip -q install 'datasets>=3.0,<5.0'
print('Repository ready:', REPO_DIR)


In [ ]:
import torch
from train import resolve_device

device = resolve_device('auto')
print('PyTorch:', torch.__version__)
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
else:
    raise RuntimeError('CUDA GPU not detected. In Colab, choose Runtime > Change runtime type > T4 GPU, then rerun this cell.')


## 1. Connect persistent storage

Colab runtimes can reset. Mount Drive so the governor history and model checkpoint survive disconnections.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUN_DIR = Path('/content/drive/MyDrive/EvolveAI')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Persistent experiment folder:', RUN_DIR)


## 2. Acquire approved learning data

This streams at most 5 MiB from the allowlisted TinyStories dataset, deduplicates records, and creates a provenance manifest. Review the upstream dataset card and license before any use beyond research.

## 2. Configure the Gatekeeper\n\nThe Gatekeeper opens approved Hugging Face sources with `streaming=True`. It does not download complete datasets or create a merged corpus. It filters short, duplicate and low-signal records, tags each record by source/domain, and maintains bounded in-memory train/validation buffers. Review each upstream dataset card and content rights before enabling its source.\n

## 3. Run one-shot training

This launches exactly one fresh model training job. It does not run a multi-trial sweep or repeated autonomous cycles. The result is saved separately in Drive; the previous governor history and champion checkpoint are preserved.

GPU quotas are enforced by Colab. A single invocation avoids repeated launches, but does not bypass the account's GPU usage limits.

In [ ]:
# ONE GPU TRAINING INVOCATION — multi-source streaming, Gatekeeper, live progress\nimport importlib, subprocess\nsubprocess.run(['git', '-C', '/content/EvolveAI', 'pull', '--ff-only'], check=True)\n\n# Reload modules in dependency order so Colab cannot retain stale imports.\nimport train; importlib.reload(train)\nimport streaming_gatekeeper; importlib.reload(streaming_gatekeeper)\nimport streaming_train; importlib.reload(streaming_train)\nimport streaming_one_shot; importlib.reload(streaming_one_shot)\nfrom streaming_one_shot import run_streaming_one_shot\n\n# Enable only after reviewing the current dataset cards and terms.\nSOURCE_IDS = ['tinystories', 'wikitext', 'ag_news', 'gsm8k']\nprint('Enabled streams:', SOURCE_IDS)\n\nreport = run_streaming_one_shot(\n    output_dir=RUN_DIR,\n    source_ids=SOURCE_IDS,\n    steps=3000,             # maximum, not guaranteed; early stopping may end sooner\n    eval_interval=100,\n    patience=5,\n    min_delta=0.002,\n    block_size=32,\n    batch_size=8,\n    hidden_size=64,\n    learning_rate=0.002,\n    seed=42,\n    architecture='baseline',\n    device='cuda',\n)\n\nprint('Mode:', report['mode'])\nprint('Steps completed:', report['steps_completed'])\nprint('Best step:', report['best_step'])\nprint('Best validation loss:', report['best_validation_loss'])\nprint('Stopped early:', report['stopped_early'])\nprint('Report:', RUN_DIR / 'streaming_one_shot_report.json')\nprint('Checkpoint:', report['checkpoint_path'])\n

In [ ]:
import json\nreport = json.loads((RUN_DIR / 'streaming_one_shot_report.json').read_text(encoding='utf-8'))\nprint('EvolveAI streaming training report')\nprint('Steps:', report['steps_completed'], '/', report['training_config']['steps_requested'])\nprint('Best validation loss:', report['best_validation_loss'])\nprint('Best step:', report['best_step'])\nprint('Elapsed seconds:', report['elapsed_seconds'])\nprint('Gatekeeper admitted:', report['gatekeeper']['records_admitted'])\nprint('Gatekeeper rejected:', report['gatekeeper']['records_rejected'])\nprint('Source statistics:')\nfor source, counts in report['gatekeeper']['sources'].items():\n    print(' ', source, counts)\nprint('Checkpoint:', report['checkpoint_path'])\n

## What this version does—and does not do\n\n- **Does:** coordinate multiple data streams, apply admission gates, keep bounded buffers, train one SNN in one invocation, print what is being trained, and save a checkpoint/report to Drive.\n- **Does not:** download complete datasets, provide unrestricted internet access, or create a structured reasoning system automatically.\n- **Validation caveat:** the validation buffer is a small online sample for monitoring, not an independent benchmark.\n- **Compute caveat:** 3,000 is a maximum step count. Early stopping can stop sooner, but GPU time is still required.\n\nA Colab notebook is a development worker, not an always-on API server. Do not expose the notebook runtime publicly with an unauthenticated tunnel.